# The Same Agent as a Graph

**What you will build:** the weather-and-calculator agent from 0.3 and 1.0 one more time — now in **LangGraph** — and for the first time you'll *see* it, as a rendered graph. That picture is the closest thing in code to n8n's canvas.

This Rosetta notebook opens Block 2: you reach for LangGraph when a task needs explicit **state, branching, cycles, or multiple agents**. For a plain tool-using agent, PydanticAI is lighter — but building this one here shows you the model.

> **Run it:** open in [Google Colab](https://colab.research.google.com/github/ezponda/ai-agents-course/blob/main/courses/python_code/book/20_rosetta_agent_as_graph.ipynb) (nothing to install), or run locally in Jupyter. Each notebook installs what it needs in its first cell.

In [ ]:
# Setup — LangChain + LangGraph on OpenRouter. Run once.
!pip install -q "langchain>=1.3,<2.0" "langgraph>=1.2,<2.0" "langchain-openai>=1.3,<2.0"

import os, getpass
from langchain_openai import ChatOpenAI

if not os.environ.get("OPENROUTER_API_KEY"):
    os.environ["OPENROUTER_API_KEY"] = getpass.getpass("Paste your OpenRouter API key: ")

MODEL_NAME = "meta-llama/llama-3.3-70b-instruct"
llm = ChatOpenAI(model=MODEL_NAME, base_url="https://openrouter.ai/api/v1",
                 api_key=os.environ["OPENROUTER_API_KEY"])
print("Ready:", MODEL_NAME)

## The agent, via `create_agent`

LangChain 1.x ships `create_agent`, which builds a ready-made ReAct agent *on the LangGraph runtime*. Tools are plain functions decorated with `@tool` (docstring = description, exactly like 1.2).

In [ ]:
from langchain_core.tools import tool
from langchain.agents import create_agent

@tool
def get_weather(city: str) -> str:
    """Get the current weather for a given city."""
    fake = {"bilbao": "18C, light rain", "madrid": "31C, sunny", "london": "14C, cloudy"}
    return fake.get(city.lower(), "20C, clear")

import ast, operator
_OPS = {ast.Add: operator.add, ast.Sub: operator.sub, ast.Mult: operator.mul, ast.Div: operator.truediv, ast.USub: operator.neg}
def safe_eval(expr):                       # same helper as notebook 0.3 -- no eval(), so 9**9**9 can't hang the kernel
    def ev(n):
        if isinstance(n, ast.Expression): return ev(n.body)
        if isinstance(n, ast.Constant) and isinstance(n.value, (int, float)): return n.value
        if isinstance(n, ast.BinOp) and type(n.op) in _OPS: return _OPS[type(n.op)](ev(n.left), ev(n.right))
        if isinstance(n, ast.UnaryOp) and type(n.op) in _OPS: return _OPS[type(n.op)](ev(n.operand))
        raise ValueError('unsupported')
    return ev(ast.parse(expr, mode='eval'))

@tool
def calculator(expression: str) -> str:
    """Evaluate a basic arithmetic expression like '12 * (3 + 4)'."""
    try:
        return str(safe_eval(expression))
    except Exception:
        return "Error: only +, -, *, / and parentheses are supported."

agent = create_agent(model=llm, tools=[get_weather, calculator],
                     system_prompt="You are a helpful assistant. Use tools when they help.")

result = agent.invoke({"messages": [{"role": "user",
          "content": "What's the weather in Madrid, and what is 231 * 17?"}]})
print(result["messages"][-1].content)

Before the picture, the proof that nothing changed underneath — print the whole message list, the way you did in 0.3 and 1.0:

In [ ]:
for m in result["messages"]:
    m.pretty_print()

User question, an AI message with `tool_calls`, tool results, final answer — the same protocol, third framework. If the Rosetta notebooks have one job, it's this: the *messages* are invariant; frameworks only change who does the bookkeeping.

Same answer again. Two things to notice about the LangGraph flavor:

- You talk to it in **messages** (`invoke({"messages": [...]})`) and read the last message's `.content`.
- What you got back, `agent`, is a compiled **graph** — which means you can draw it.

## See the graph

This is the payoff. `create_agent` built a small state machine; let's render it.

In [ ]:
graph = agent
from IPython.display import Image, display
try:
    display(Image(graph.get_graph().draw_mermaid_png()))     # PNG via mermaid.ink (needs internet)
except Exception:
    print("Mermaid image unavailable, showing text instead:\n")
    print(graph.get_graph().draw_mermaid())                  # pure text (Mermaid DSL), no extra deps

You're looking at the ReAct loop from 0.3, drawn: an agent node that either calls **tools** and loops back, or finishes. PydanticAI ran this same loop but kept it invisible; LangGraph makes the structure a first-class, inspectable object. That visibility is exactly what you want once the flow gets complicated — which is the rest of Block 2.

| | PydanticAI (Block 1) | LangGraph (Block 2) |
|--|----------------------|---------------------|
| Best at | one clean, typed agent | explicit state, branching, cycles, multi-agent |
| The flow is | implicit (hidden loop) | an explicit graph you can draw and edit |
| You write | functions + decorators | nodes + edges + state |

## The real case for a graph (the picture is the least of it)

A drawing is nice; nobody adopts a runtime for a drawing. The engineering reasons come from the graph being an **explicit, stateful machine** rather than a hidden loop — each of these gets its own notebook:

1. **Persistence you attach, not thread through.** One argument (`checkpointer=`) and every step of every conversation is saved and resumable — no plumbing through your functions (**2.3**).
2. **Pause and resume across processes.** A graph can stop mid-run — waiting for a human — and continue *days later, in a different process*. That's the save-and-resume problem from 0.5, solved at the runtime level (**2.4**).
3. **Time travel.** Saved steps mean you can rewind to any point, inspect the state, and re-run from there — a debugger for agent flows (**2.3**).
4. **Cycles with visible stop conditions.** Loops (reflection, retry-until-good) become edges you can see and cap, instead of `while` logic buried in code (**2.5**).
5. **Parallel branches with controlled merging.** Fan out, fan in, without hand-rolled `asyncio` — the state itself defines how concurrent results combine (**2.1**).

Keep the flip side from the table in view: for a *plain* tool-using agent, none of these apply and PydanticAI stays the lighter tool. Block 2 is about recognizing which side of that line a task falls on.

```{note}
`create_agent` is the convenient prebuilt. To actually *use* graphs — custom state, your own branching and loops — you build the graph yourself. That's the next notebook, and it's where LangGraph earns its place.
```

::::{dropdown} 🛠️ Try it yourself
:color: secondary

1. **Add a third tool** (`word_count`, as in 1.0) and re-render the graph. **Done when:** you can explain why the drawing *didn't change*.
2. **Read the Mermaid.** Run `print(graph.get_graph().draw_mermaid())` and match every line of the DSL to the picture. **Done when:** nothing in the text surprises you.
3. **Trace a no-tool question.** Ask "Say hi." and print the messages. **Done when:** you've confirmed the tools node never ran (the conditional edge went straight to END).
::::

::::{dropdown} 🛠️ Solutions
:color: secondary

**1 —** The graph is `model ⇄ tools`: the tools node is *one* node that runs whatever the model requested, however many tools are registered. Adding a tool changes the model's options (the schemas in the request — 0.3), not the control-flow topology. Structure and capability are different axes: the graph draws the first.

**3 —**

```python
out = graph.invoke({"messages": [{"role": "user", "content": "Say hi."}]})
for m in out["messages"]:
    m.pretty_print()          # user, then one AIMessage — no tool messages anywhere
```

Two messages total: the conditional edge saw no `tool_calls` and finished. The cycle only exists when taken.
::::

**What's next:** in **2.1** we build a graph from scratch — nodes, edges, state — so the picture above stops being magic.